In [1]:
import tensorflow as tf

print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))
print(tf.test.is_built_with_cuda())

I0000 00:00:1778876517.970276   58030 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2.21.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
True


In [2]:
import os
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, Input, mixed_precision
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import (
    ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
)

In [3]:
# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
IMG_SIZE   = 256
BATCH_SIZE = 4        
EPOCHS     = 100
DATA_PATH  = '/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2'
MODEL_SAVE = '/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/unet-trsenet/depth-se/multitask_polypgen_best.keras'

In [ ]:

# ─────────────────────────────────────────────
# POLYPGEN DATA LOADER
# ─────────────────────────────────────────────
def load_polypgen_data(base_path):
    images_list = []
    masks_list = []
    labels_list = []

    # Loading Multicenter Positive data from Data_C1 to Data_C6
    for i in range(1, 7):
        c_img_dir = os.path.join(base_path, f'data_C{i}', f'images_C{i}')
        c_mask_dir = os.path.join(base_path, f'data_C{i}', f'masks_C{i}')
        
        if os.path.exists(c_img_dir):
            imgs = sorted(glob.glob(os.path.join(c_img_dir, '*.jpg')))
            msks = sorted(glob.glob(os.path.join(c_mask_dir, '*.jpg')))
            images_list.extend(imgs)
            masks_list.extend(msks)
            labels_list.extend([1.0] * len(imgs)) # Polyp exist = 1.0

    # 2 Sequence Positive data
    pos_seq_dir = os.path.join(base_path, 'sequenceData', 'positive')
    if os.path.exists(pos_seq_dir):
        for seq_folder in os.listdir(pos_seq_dir):
            img_dir = os.path.join(pos_seq_dir, seq_folder, f'images_{seq_folder}')
            mask_dir = os.path.join(pos_seq_dir, seq_folder, f'masks_{seq_folder}')
            if os.path.exists(img_dir):
                imgs = sorted(glob.glob(os.path.join(img_dir, '*.jpg')))
                msks = sorted(glob.glob(os.path.join(mask_dir, '*.jpg')))
                images_list.extend(imgs)
                masks_list.extend(msks)
                labels_list.extend([1.0] * len(imgs))

    print(f"Total Positive (Polyp) Samples: {len(images_list)}")

    # 3. Collecting Negative data (sequenceData/negativeOnly)
    neg_seq_dir = os.path.join(base_path, 'sequenceData', 'negativeOnly')
    neg_images = []
    if os.path.exists(neg_seq_dir):
        for seq_folder in os.listdir(neg_seq_dir):
            img_dir = os.path.join(neg_seq_dir, seq_folder)
            if os.path.exists(img_dir):
                imgs = sorted(glob.glob(os.path.join(img_dir, '*.jpg')))
                neg_images.extend(imgs)
                
    # Data balancing
    np.random.seed(42)
    if len(neg_images) > len(images_list):
        neg_images = np.random.choice(neg_images, size=len(images_list), replace=False).tolist()

    for n_img in neg_images:
        images_list.append(n_img)
        masks_list.append('EMPTY_MASK') 
        labels_list.append(0.0)     

    print(f"Total Combined Samples (Polyp + Non-Polyp): {len(images_list)}")
    return images_list, masks_list, labels_list

In [5]:

# ─────────────────────────────────────────────
#  AUGMENTATION 
# ─────────────────────────────────────────────
def augment(img, outputs_dict):
    mask = outputs_dict['seg_out']
    label = outputs_dict['clf_out']
    
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    
    return img, {'seg_out': mask, 'clf_out': label}

In [ ]:
# ─────────────────────────────────────────────
#  PREPROCESSING
# ─────────────────────────────────────────────
def process_path(img_path, mask_path, label):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0


    def load_real_mask(m_path):
        m = tf.io.read_file(m_path)
        m = tf.image.decode_image(m, channels=1, expand_animations=False)
        m = tf.image.resize(m, [IMG_SIZE, IMG_SIZE], method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
        m = tf.cast(m, tf.float32) / 255.0
        return tf.where(m > 0.5, 1.0, 0.0)

    def generate_empty_mask():
        return tf.zeros([IMG_SIZE, IMG_SIZE, 1], dtype=tf.float32)

    mask = tf.cond(tf.equal(mask_path, 'EMPTY_MASK'), 
                   generate_empty_mask, 
                   lambda: load_real_mask(mask_path))

    return img, {'seg_out': mask, 'clf_out': label}

def get_dataset(x, y, labels, batch=BATCH_SIZE, augment_data=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y, labels))
    ds = ds.shuffle(buffer_size=len(x), reshuffle_each_iteration=True)
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

In [7]:

# ─────────────────────────────────────────────
#  LOSS & METRICS
# ─────────────────────────────────────────────
def dice_coef(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
    )

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    y_true_f = tf.cast(y_true, tf.float32)
    y_pred_f = tf.cast(y_pred, tf.float32)
    
    # Binary cross entropy is applicable for all images
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true_f, y_pred_f))
    
    # Conditional dice lose calculation if exist polyp then calculate it
    # y_true summation > 0 means polyp exist
    has_polyp = tf.reduce_sum(y_true_f) > 0
    
    loss = tf.cond(
        has_polyp,
        lambda: bce + dice_loss(y_true_f, y_pred_f), # If polyp exist then BCE + Dice
        lambda: bce # if polyp exist then BCE, otherwise no dice loss
    )
    return loss

def iou_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
    return (intersection + 1.0) / (union + 1.0)


In [8]:

# ─────────────────────────────────────────────
#  MODULES & CUSTOM BLOCKS
# ─────────────────────────────────────────────
def se_block(input_tensor, ratio=16):
    """Squeeze-and-Excitation Block"""
    channels = input_tensor.shape[-1]
    
    # Squeeze: Global Information Capture
    # (Batch, H, W, C) -> (Batch, 1, 1, C)
    squeeze = layers.GlobalAveragePooling2D()(input_tensor)
    
    # Excitation: Learning channel-wise weights
    excitation = layers.Dense(channels // ratio, activation='relu', use_bias=False)(squeeze)
    excitation = layers.Dense(channels, activation='sigmoid', use_bias=False)(excitation)
    
    # Reshape for multiplication
    excitation = layers.Reshape((1, 1, channels))(excitation)
    
    # Scale: Multiply original input by learned weights
    return layers.multiply([input_tensor, excitation])

def depthwise_seperable_conv_block(x, filters, dropout_rate=0.0):
    x = layers.SeparableConv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    if dropout_rate > 0:
        x = layers.SpatialDropout2D(dropout_rate)(x)
        
    x = layers.SeparableConv2D(filters, 5, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.SeparableConv2D(filters, 7, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = se_block(x) 
    return x


In [9]:
def build_unet_multitask(input_shape=(256, 256, 3)):
    inputs = Input(input_shape)

    # ── ENCODER  ──
    c1 = depthwise_seperable_conv_block(inputs, 32)
    p1 = layers.MaxPooling2D(2)(c1)

    c2 = depthwise_seperable_conv_block(p1, 64)
    p2 = layers.MaxPooling2D(2)(c2)

    c3 = depthwise_seperable_conv_block(p2, 128, dropout_rate=0.1)
    p3 = layers.MaxPooling2D(2)(c3)

    c4 = depthwise_seperable_conv_block(p3, 256, dropout_rate=0.2)
    p4 = layers.MaxPooling2D(2)(c4)

    # ── BOTTLENECK Strongest Layer ──
    b = depthwise_seperable_conv_block(p4, 512, dropout_rate=0.3)

    # ──  CLASSIFICATION BRANCH  ──
    # To classify it will take the features for classification
    cls_layer = layers.GlobalAveragePooling2D()(b)
    cls_layer = layers.Dense(128, activation='relu')(cls_layer)
    cls_layer = layers.Dropout(0.4)(cls_layer)
    cls_out = layers.Dense(1, activation='sigmoid', name='clf_out', dtype='float32')(cls_layer)

    # ── DECODER  ──
    u4 = layers.Conv2DTranspose(256, 2, strides=2, padding='same')(b)
    u4 = layers.concatenate([u4, c4])
    c5 = depthwise_seperable_conv_block(u4, 256, dropout_rate=0.2)

    u3 = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(c5)
    u3 = layers.concatenate([u3, c3])
    c6 = depthwise_seperable_conv_block(u3, 128, dropout_rate=0.1)

    u2 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(c6)
    u2 = layers.concatenate([u2, c2])
    c7 = depthwise_seperable_conv_block(u2, 64)

    u1 = layers.Conv2DTranspose(32, 2, strides=2, padding='same')(c7)
    u1 = layers.concatenate([u1, c1])
    c8 = depthwise_seperable_conv_block(u1, 32)

    # ──  SEGMENTATION BRANCH ──
    seg_out = layers.Conv2D(1, 1, activation='sigmoid', name='seg_out', dtype='float32')(c8)

    return models.Model(inputs=inputs, outputs=[seg_out, cls_out], name='PolypGen-MultiTask-NetX')

In [ ]:

# ─────────────────────────────────────────────
#  MAIN RUNNER
# ─────────────────────────────────────────────
if __name__ == '__main__':
    # PolypGen, Dataset path Loading
    images, masks, labels = load_polypgen_data(DATA_PATH)

    # Data Spliting and convert it into two set
    train_x, val_x, train_y, val_y, train_l, val_l = train_test_split(
        images, masks, labels, test_size=0.2, random_state=42, stratify=labels
    )
    print(f"Train samples: {len(train_x)}  |  Val samples: {len(val_x)}")

    # Generator Calling
    train_ds = get_dataset(train_x, train_y, train_l, batch=BATCH_SIZE, augment_data=True)
    val_ds   = get_dataset(val_x,   val_y,   val_l,   batch=BATCH_SIZE, augment_data=False)

    # Model Initialization
    unet_multitask = build_unet_multitask()
    unet_multitask.summary()

    # Optimizer settings
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
    optimizer = mixed_precision.LossScaleOptimizer(optimizer)

    # Multi-task model compiling
    unet_multitask.compile(
        optimizer=optimizer,
        loss={
            'seg_out': bce_dice_loss,           # Segmentation Loss
            'clf_out': 'binary_crossentropy'    # Classification Loss
        },
        loss_weights={
            'seg_out': 1.0,   # Segmentation got the main priority
            'clf_out': 0.2   # less weight for classification loss
        },
        metrics={
            'seg_out': [dice_coef, iou_metric, 'accuracy'],
            'clf_out': 'accuracy'  #Measure accuricy of classification
        }
    )

    os.makedirs(os.path.dirname(MODEL_SAVE), exist_ok=True)

    # 
    callbacks = [
        ModelCheckpoint(MODEL_SAVE, monitor='val_seg_out_dice_coef',
                        save_best_only=True, mode='max', verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=7, min_lr=1e-7, verbose=1),
        EarlyStopping(monitor='val_seg_out_dice_coef', patience=20,
                      restore_best_weights=True, mode='max', verbose=1),
    ]

    print("\n========== Training Started ==========")
    history = unet_multitask.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks
    )
    print("========== Training Complete ==========\n")

    # ── Plot Results ────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Total Loss Plot
    axes[0].plot(history.history['loss'],          label='Train Total')
    axes[0].plot(history.history['val_loss'],      label='Val Total')
    axes[0].set_title('Multi-Task Total Loss'); axes[0].legend()

    # Segmentation Dice Plot
    axes[1].plot(history.history['seg_out_dice_coef'],     label='Train Seg Dice')
    axes[1].plot(history.history['val_seg_out_dice_coef'], label='Val Seg Dice')
    axes[1].set_title('Segmentation Dice Coefficient'); axes[1].legend()

    # Classification accuricy plot
    axes[2].plot(history.history['clf_out_accuracy'],     label='Train Clf Acc')
    axes[2].plot(history.history['val_clf_out_accuracy'], label='Val Clf Acc')
    axes[2].set_title('Classification Accuracy'); axes[2].legend()

    plt.tight_layout()
    plt.savefig('multi_task_training_history.png', dpi=150)
    plt.show()

Total Positive (Polyp) Samples: 3762
Total Combined Samples (Polyp + Non-Polyp): 6282
Train samples: 5025  |  Val samples: 1257


W0000 00:00:1778876669.318481   58030 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1778876669.319596   58030 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "PolypGen-MultiTask-NetX"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d    │ (None, 256, 256,  │        123 │ input_layer[0][0] │
│ (SeparableConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256, 256,  │        128 │ separable_conv2d… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_1  │ (None, 256, 256,  │      1,824 │ activation[0][0]  │
│ (SeparableConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ separable_conv2d… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_2  │ (None, 256, 256,  │      2,592 │ activation_1[0][… │
│ (SeparableConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ separable_conv2d… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ activation_2[0][… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 2)         │         64 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │         64 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1, 32)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 256, 256,  │          0 │ activation_2[0][… │
│                     │ 32)               │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ multiply[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_3  │ (None, 128, 128,  │      2,336 │ max_pooling2d[0]… │
│ (SeparableConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ separable_conv2d

 Total params: 2,177,437 (8.31 MB)

...


========== Training Started ==========
Epoch 1/100


I0000 00:00:1778876692.066877   63596 service.cc:153] XLA service 0x7f11440871f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778876692.066986   63596 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 12.7.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.21.1)
I0000 00:00:1778876692.985817   63596 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778876698.277464   63596 cuda_dnn.cc:461] Loaded cuDNN version 92101
I0000 00:00:1778876699.495728   63596 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_32767__.407
E0000 00:00:1778876722.283829   63596 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1778876723.070252   63596 cuda_timer.cc:87] Delay kernel time

1256/1257 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - clf_out_accuracy: 0.6361 - clf_out_loss: 0.6487 - loss: 1.2530 - seg_out_accuracy: 0.9380 - seg_out_dice_coef: 0.0736 - seg_out_iou_metric: 0.0509 - seg_out_loss: 1.1232

E0000 00:00:1778876899.667435   63595 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1778876915.964741   63595 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


1257/1257 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - clf_out_accuracy: 0.6361 - clf_out_loss: 0.6487 - loss: 1.2529 - seg_out_accuracy: 0.9380 - seg_out_dice_coef: 0.0736 - seg_out_iou_metric: 0.0509 - seg_out_loss: 1.1232
Epoch 1: val_seg_out_dice_coef improved from None to 0.07215, saving model to /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/unet-trsenet/depth-se/multitask_polypgen_best.keras

Epoch 1: finished saving model to /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/unet-trsenet/depth-se/multitask_polypgen_best.keras
1257/1257 ━━━━━━━━━━━━━━━━━━━━ 294s 147ms/step - clf_out_accuracy: 0.6577 - clf_out_loss: 0.6089 - loss: 1.1736 - seg_out_accuracy: 0.9536 - seg_out_dice_coef: 0.0790 - seg_out_iou_metric: 0.0630 - seg_out_loss: 1.0517 - val_clf_out_accuracy: 0.3134 - val_clf_out_loss: 0.6956 - val_loss: 1.1640 - val_seg_out_accuracy: 0.9590 - val_seg_out_dice_coef: 0.0721 - val_seg_out_iou_metric: 0.0896 - val_seg_out_loss: 1.0254 - learn